In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os
import tensorflow as tf
from PIL import Image

train = np.loadtxt(
    "../emnist-byclass-train.csv",
    delimiter=",",
    skiprows=1,
    dtype=np.uint8
)

test = np.loadtxt(
    "../emnist-byclass-test.csv",
    delimiter=",",
    skiprows=1,
    dtype=np.uint8
)

I0000 00:00:1779966109.692162    7865 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779966109.695905    7865 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779966110.117058    7865 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779966111.142985    7865 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [2]:
def fix_emnist_letters_orientation(X_letters):
    X_letters = X_letters.reshape(-1, 28, 28)
    fixed_images = []
    for img in X_letters:
        pil_img = Image.fromarray(img.astype(np.uint8))
        pil_img = pil_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        pil_img = pil_img.rotate(90, expand=False)
        fixed_images.append(np.array(pil_img))
    return np.array(fixed_images).reshape(-1, 784)


In [3]:
X_train = train[:, 1:].astype(np.float32)
y_train = train[:, 0].astype(np.int64)

X_test = test[:, 1:].astype(np.float32)
y_test = test[:, 0].astype(np.int64)

In [4]:
X_train = fix_emnist_letters_orientation(X_train)
X_test = fix_emnist_letters_orientation(X_test)

In [5]:
X_train = X_train.astype(np.float32)
X_train /= 255.0

X_test = X_test.astype(np.float32)
X_test /= 255.0

In [6]:
y_train_onehot = tf.keras.utils.to_categorical(
    y_train,
    num_classes=62
)

y_test_onehot = tf.keras.utils.to_categorical(
    y_test,
    num_classes=62
)

In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),

    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(62, activation="softmax")
])

E0000 00:00:1779966157.257202    7865 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779966157.257809    8220 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1779966157.274040    7865 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [8]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 574,142 (2.19 MB)

 Trainable params: 574,142 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [10]:

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_test, y_test_onehot),
    epochs=40,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.7540 - loss: 0.7940 - val_accuracy: 0.8239 - val_loss: 0.5071
Epoch 2/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8037 - loss: 0.5954 - val_accuracy: 0.8342 - val_loss: 0.4708
Epoch 3/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8127 - loss: 0.5612 - val_accuracy: 0.8410 - val_loss: 0.4527
Epoch 4/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8177 - loss: 0.5437 - val_accuracy: 0.8436 - val_loss: 0.4397
Epoch 5/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8200 - loss: 0.5333 - val_accuracy: 0.8445 - val_loss: 0.4358
Epoch 6/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.8222 - loss: 0.5264 - val_accuracy: 0.8458 - val_loss: 0.4325
Epoch 7/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8236 - loss: 0.5191 - val_accuracy: 0.8466 - val_loss: 0.4335
Epoch 8/40
5453/5453 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8252 - loss: 0

In [12]:
model.save("../trained_models/ann_model.keras")

print("Model saved!")

Model saved!
